# 📈 **S&P 500 Data Pipeline**: Comprehensive Data Management for Predictive Analytics

# Imports and setup

In [ ]:
import os
import sys
import json
import time
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Any
import requests
from pathlib import Path
import pickle
import hashlib
from functools import wraps

# Add parent to path for imports
sys.path.append('..')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Imports loaded successfully")

✅ Imports loaded successfully


# Configuration

In [ ]:
class DataConfig:
    """Data collection configuration"""
    
    # API Keys (set in .env file)
    ALPHA_VANTAGE_KEY = os.getenv('ALPHA_VANTAGE_KEY', 'YOUR_API_KEY')
    FRED_API_KEY = os.getenv('FRED_API_KEY', 'YOUR_API_KEY')
    
    # Date ranges
    START_DATE = '2010-01-01'
    END_DATE = datetime.now().strftime('%Y-%m-%d')
    
    # Tickers
    SP500_TICKER = 'SPY'
    VIX_TICKER = '^VIX'
    
    # FRED Series IDs
    FRED_SERIES = {
        'DGS10': '10yr_treasury_rate',
        'FEDFUNDS': 'fed_funds_rate',
        'UNRATE': 'unemployment_rate',
        'CPIAUCSL': 'cpi',
        'GDP': 'gdp',
        'SP500': 'sp500_index'
    }
    
    # Cache settings
    CACHE_DIR = Path('../../data/cache')
    CACHE_DURATION_HOURS = 6
    
    # Rate limiting
    ALPHA_VANTAGE_DELAY = 12

print("✅ Configuration loaded")
print(f"📅 Date range: {DataConfig.START_DATE} to {DataConfig.END_DATE}")

✅ Configuration loaded
📅 Date range: 2010-01-01 to 2026-04-10


# Yahoo Finance Collector

In [ ]:
class YahooFinanceCollector:
    """Collect S&P 500 data from Yahoo Finance"""
    
    def __init__(self):
        self.ticker = yf.Ticker(DataConfig.SP500_TICKER)
        
    def fetch_daily_data(self, start_date: str, end_date: str) -> pd.DataFrame:
        """Fetch daily OHLCV data"""
        print(f"📊 Fetching S&P 500 data from Yahoo Finance...")
        
        try:
            data = self.ticker.history(start=start_date, end=end_date)
            
            # Clean column names
            data.columns = [col.lower() for col in data.columns]
            
            # Add additional columns
            data['returns'] = data['close'].pct_change()
            data['log_returns'] = np.log(data['close'] / data['close'].shift(1))
            data['volatility'] = data['returns'].rolling(window=20).std() * np.sqrt(252)
            
            print(f"✅ Fetched {len(data)} days of data")
            return data
            
        except Exception as e:
            print(f"❌ Error fetching Yahoo data: {e}")
            return pd.DataFrame()
    
    def fetch_intraday_data(self, period: str = '1mo', interval: str = '1h') -> pd.DataFrame:
        """Fetch intraday data (hourly)"""
        print(f"📊 Fetching intraday data with {interval} interval...")
        
        try:
            data = self.ticker.history(period=period, interval=interval)
            data.columns = [col.lower() for col in data.columns]
            return data
            
        except Exception as e:
            print(f"❌ Error fetching intraday data: {e}")
            return pd.DataFrame()

# Test the collector
yahoo_collector = YahooFinanceCollector()
print("✅ Yahoo Finance Collector ready")

✅ Yahoo Finance Collector ready


# Alpha Vantage Collector

In [ ]:
class AlphaVantageCollector:
    """Collect technical indicators from Alpha Vantage"""
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = 'https://www.alphavantage.co/query'
        self.delay = DataConfig.ALPHA_VANTAGE_DELAY
        
    def _make_request(self, params: Dict) -> Optional[Dict]:
        """Make API request with rate limiting"""
        time.sleep(self.delay)
        
        try:
            response = requests.get(self.base_url, params=params)
            if response.status_code == 200:
                return response.json()
            else:
                print(f"❌ API error: {response.status_code}")
                return None
        except Exception as e:
            print(f"❌ Request error: {e}")
            return None
    
    def fetch_rsi(self, symbol: str, interval: str = 'daily', time_period: int = 14) -> pd.DataFrame:
        """Fetch RSI indicator"""
        params = {
            'function': 'RSI',
            'symbol': symbol,
            'interval': interval,
            'time_period': time_period,
            'apikey': self.api_key
        }
        
        data = self._make_request(params)
        if data and 'Technical Analysis: RSI' in data:
            df = pd.DataFrame.from_dict(data['Technical Analysis: RSI'], orient='index')
            df.index = pd.to_datetime(df.index)
            return df.sort_index()
        
        return pd.DataFrame()
    
    def fetch_macd(self, symbol: str, interval: str = 'daily') -> pd.DataFrame:
        """Fetch MACD indicator"""
        params = {
            'function': 'MACD',
            'symbol': symbol,
            'interval': interval,
            'apikey': self.api_key
        }
        
        data = self._make_request(params)
        if data and 'Technical Analysis: MACD' in data:
            df = pd.DataFrame.from_dict(data['Technical Analysis: MACD'], orient='index')
            df.index = pd.to_datetime(df.index)
            return df.sort_index()
        
        return pd.DataFrame()

# Initialize (use 'demo' for testing)
alpha_collector = AlphaVantageCollector(os.getenv('ALPHA_VANTAGE_KEY', 'demo'))
print("✅ Alpha Vantage Collector ready")

✅ Alpha Vantage Collector ready


# FRED Economic Data Collector

In [ ]:
class FREDCollector:
    """Collect economic indicators from FRED API"""
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = 'https://api.stlouisfed.org/fred/series/observations'
        
    def fetch_series(self, series_id: str, start_date: str, end_date: str) -> pd.DataFrame:
        """Fetch economic time series from FRED"""
        print(f"📊 Fetching FRED series: {series_id}...")
        
        params = {
            'series_id': series_id,
            'api_key': self.api_key,
            'file_type': 'json',
            'observation_start': start_date,
            'observation_end': end_date
        }
        
        try:
            response = requests.get(self.base_url, params=params)
            
            if response.status_code == 200:
                data = response.json()
                observations = data['observations']
                
                df = pd.DataFrame(observations)
                df['date'] = pd.to_datetime(df['date'])
                df['value'] = pd.to_numeric(df['value'], errors='coerce')
                df = df[['date', 'value']].set_index('date')
                df.columns = [DataConfig.FRED_SERIES.get(series_id, series_id)]
                
                return df
            else:
                print(f"❌ FRED API error: {response.status_code}")
                return pd.DataFrame()
                
        except Exception as e:
            print(f"❌ Error fetching FRED data: {e}")
            return pd.DataFrame()
    
    def fetch_all_indicators(self, start_date: str, end_date: str) -> pd.DataFrame:
        """Fetch all configured FRED indicators"""
        all_data = []
        
        for series_id in DataConfig.FRED_SERIES.keys():
            df = self.fetch_series(series_id, start_date, end_date)
            if not df.empty:
                all_data.append(df)
            time.sleep(1)
        
        if all_data:
            combined = pd.concat(all_data, axis=1)
            combined = combined.ffill()
            return combined
        
        return pd.DataFrame()

fred_collector = FREDCollector(os.getenv('FRED_API_KEY', 'demo'))
print("✅ FRED Collector ready")

✅ FRED Collector ready


### Step 2: Data Cleaner (Handle NaN, outliers, forward fill)

# Data Cleaner (FIXED - no aggressive outlier removal)

In [ ]:
class DataCleaner:
    """Clean and preprocess financial data"""
    
    def __init__(self, config: Optional[Dict] = None):
        self.config = config or {
            'outlier_method': 'cap',  # CHANGED: 'cap' instead of 'zscore'
            'zscore_threshold': 5,     # INCREASED: 3 -> 5 (less aggressive)
            'iqr_multiplier': 3.0,     # INCREASED: 1.5 -> 3.0
            'fill_method': 'ffill',
            'max_nan_pct': 0.3,
            'remove_outliers': False   # NEW: Don't remove, just cap
        }
    
    def clean_sp500_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """Complete cleaning pipeline"""
        print("🧹 Starting data cleaning pipeline...")
        
        df_clean = df.copy()
        
        # Remove duplicates
        before_len = len(df_clean)
        df_clean = df_clean[~df_clean.index.duplicated(keep='first')]
        print(f"  - Removed {before_len - len(df_clean)} duplicate rows")
        
        # Drop columns with too many NaNs
        nan_pct = df_clean.isnull().sum() / len(df_clean)
        cols_to_drop = nan_pct[nan_pct > self.config['max_nan_pct']].index
        if len(cols_to_drop) > 0:
            df_clean = df_clean.drop(columns=cols_to_drop)
            print(f"  - Dropped columns: {list(cols_to_drop)}")
        
        # Fill missing values
        df_clean = self._fill_missing_values(df_clean)
        
        # Handle outliers (CAP them, don't remove)
        df_clean = self._handle_outliers(df_clean)
        
        # Validate ranges
        df_clean = self._validate_ranges(df_clean)
        
        print(f"✅ Cleaning complete. Shape: {df_clean.shape}")
        return df_clean
    
    def _fill_missing_values(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fill missing values using modern pandas methods"""
        
        if self.config['fill_method'] == 'ffill':
            df = df.ffill()
            df = df.bfill()
        elif self.config['fill_method'] == 'linear':
            df = df.interpolate(method='linear', limit_direction='both')
        elif self.config['fill_method'] == 'interpolate':
            df = df.interpolate(method='time', limit_direction='both')
        
        return df
    
    def _handle_outliers(self, df: pd.DataFrame) -> pd.DataFrame:
        """Cap outliers instead of removing rows"""
        
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        
        # Skip returns and log_returns as they naturally have outliers
        cols_to_skip = ['returns', 'log_returns', 'volatility']
        numeric_cols = [col for col in numeric_cols if col not in cols_to_skip]
        
        if self.config['outlier_method'] == 'cap':
            # Cap outliers using percentile method (more robust)
            for col in numeric_cols:
                if col in df.columns:
                    # Use 1st and 99th percentiles for capping
                    lower_cap = df[col].quantile(0.01)
                    upper_cap = df[col].quantile(0.99)
                    
                    before_count = ((df[col] < lower_cap) | (df[col] > upper_cap)).sum()
                    
                    # Cap the values
                    df[col] = df[col].clip(lower=lower_cap, upper=upper_cap)
                    
                    if before_count > 0:
                        print(f"  - Capped {before_count} outliers in {col}")
            
        elif self.config['outlier_method'] == 'zscore':
            # Less aggressive zscore - only remove extreme outliers
            try:
                from scipy import stats
                for col in numeric_cols:
                    if col in df.columns:
                        z_scores = np.abs(stats.zscore(df[col].fillna(df[col].mean())))
                        outliers = z_scores > self.config['zscore_threshold']
                        if outliers.any():
                            # Cap instead of remove
                            mean_val = df[col].mean()
                            std_val = df[col].std()
                            upper_limit = mean_val + self.config['zscore_threshold'] * std_val
                            lower_limit = mean_val - self.config['zscore_threshold'] * std_val
                            df.loc[df[col] > upper_limit, col] = upper_limit
                            df.loc[df[col] < lower_limit, col] = lower_limit
                            print(f"  - Capped {outliers.sum()} outliers in {col}")
            except:
                print(f"  - scipy not available, skipping outlier detection")
        
        return df
    
    def _validate_ranges(self, df: pd.DataFrame) -> pd.DataFrame:
        """Validate and fix data ranges"""
        # Ensure price columns are positive
        price_cols = ['open', 'high', 'low', 'close', 'adj close']
        for col in price_cols:
            if col in df.columns:
                df[col] = df[col].clip(lower=0)
        
        # Ensure high >= low
        if 'high' in df.columns and 'low' in df.columns:
            invalid = df['high'] < df['low']
            if invalid.any():
                df.loc[invalid, 'high'] = df.loc[invalid, 'low'] * 1.001
                print(f"  - Fixed {invalid.sum()} high<low violations")
        
        # Ensure close is within high-low range
        if all(col in df.columns for col in ['high', 'low', 'close']):
            invalid_high = df['close'] > df['high']
            invalid_low = df['close'] < df['low']
            if invalid_high.any():
                df.loc[invalid_high, 'close'] = df.loc[invalid_high, 'high']
            if invalid_low.any():
                df.loc[invalid_low, 'close'] = df.loc[invalid_low, 'low']
            if invalid_high.any() or invalid_low.any():
                print(f"  - Fixed {invalid_high.sum() + invalid_low.sum()} close range violations")
        
        return df

cleaner = DataCleaner()
print("✅ Data Cleaner ready (FIXED - no row deletion)")

✅ Data Cleaner ready (FIXED - no row deletion)


# Clear cache and re-run

In [ ]:
cache_manager.clear()  # Clear old cached data
print("Cache cleared. Now re-run the pipeline.")

🗑️ Cleared 0 cache files
Cache cleared. Now re-run the pipeline.


### Step 3: Data Validator (Schema checks, data quality)

# Data Validator (UPDATED - no deprecated methods)

In [ ]:
class DataValidator:
    """Validate data quality and schema compliance"""
    
    def __init__(self):
        self.schema = {
            'required_columns': ['open', 'high', 'low', 'close', 'volume'],
            'data_types': {
                'open': float, 'high': float, 'low': float, 
                'close': float, 'volume': int
            },
            'value_ranges': {
                'open': (0, None), 'high': (0, None), 'low': (0, None),
                'close': (0, None), 'volume': (0, None),
                'returns': (-0.2, 0.2),
                'volatility': (0, 1)
            }
        }
    
    def generate_quality_report(self, df: pd.DataFrame) -> Dict:
        """Generate comprehensive data quality report"""
        print("📋 Generating quality report...")
        
        # Completeness check
        missing_pct = (df.isnull().sum() / len(df) * 100).to_dict()
        
        # Convert numpy types to Python types for JSON serialization
        missing_pct_clean = {}
        for key, value in missing_pct.items():
            if isinstance(value, (np.float32, np.float64)):
                missing_pct_clean[key] = float(value)
            else:
                missing_pct_clean[key] = value
        
        # Duplicate check
        duplicate_rows = int(df.duplicated().sum())
        
        # Range validation
        range_errors = {}
        for col, (min_val, max_val) in self.schema['value_ranges'].items():
            if col in df.columns:
                errors = {}
                if min_val is not None and (df[col] < min_val).any():
                    errors['below_min'] = int((df[col] < min_val).sum())
                if max_val is not None and (df[col] > max_val).any():
                    errors['above_max'] = int((df[col] > max_val).sum())
                if errors:
                    range_errors[col] = errors
        
        # Check required columns
        missing_columns = [col for col in self.schema['required_columns'] if col not in df.columns]
        
        # Calculate quality score
        score = 100.0
        
        # Deduct for missing values
        if missing_pct_clean:
            avg_missing = float(np.mean(list(missing_pct_clean.values())))
            score -= avg_missing * 0.5
        
        # Deduct for duplicates
        if duplicate_rows > 0:
            score -= min(20.0, (duplicate_rows / len(df)) * 100)
        
        # Deduct for range errors
        if range_errors:
            score -= len(range_errors) * 5.0
        
        # Deduct for missing columns
        if missing_columns:
            score -= len(missing_columns) * 10.0
        
        report = {
            'quality_score': max(0.0, score),
            'is_valid': score >= 70,
            'missing_percentage': missing_pct_clean,
            'duplicate_rows': duplicate_rows,
            'range_errors': range_errors,
            'missing_required_columns': missing_columns,
            'total_rows': len(df),
            'total_columns': len(df.columns)
        }
        
        print(f"  - Quality score: {report['quality_score']:.1f}/100")
        print(f"  - Valid for modeling: {report['is_valid']}")
        print(f"  - Total rows: {report['total_rows']}")
        print(f"  - Total columns: {report['total_columns']}")
        
        if missing_columns:
            print(f"  ⚠️ Missing columns: {missing_columns}")
        
        return report

validator = DataValidator()
print("✅ Data Validator ready (UPDATED)")

✅ Data Validator ready (UPDATED)


### Step 4: Cache Manager (Avoid redundant API calls)

# Cache Manager (FIXED - with decorator method)

In [ ]:
class CacheManager:
    """Manage caching of API responses"""
    
    def __init__(self, cache_dir: Path = DataConfig.CACHE_DIR, 
                 cache_duration_hours: int = DataConfig.CACHE_DURATION_HOURS):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.cache_duration = timedelta(hours=cache_duration_hours)
        
    def _get_cache_key(self, *args, **kwargs) -> str:
        """Generate unique cache key"""
        key_data = f"{args}{sorted(kwargs.items())}"
        return hashlib.md5(key_data.encode()).hexdigest()
    
    def _get_cache_path(self, cache_key: str) -> Path:
        """Get file path for cache key"""
        return self.cache_dir / f"{cache_key}.pkl"
    
    def get(self, func_name: str, *args, **kwargs) -> Optional[Any]:
        """Retrieve cached data"""
        cache_key = self._get_cache_key(func_name, *args, **kwargs)
        cache_path = self._get_cache_path(cache_key)
        
        if cache_path.exists():
            age = datetime.now() - datetime.fromtimestamp(cache_path.stat().st_mtime)
            if age < self.cache_duration:
                with open(cache_path, 'rb') as f:
                    print(f"💾 Cache HIT for {func_name}")
                    return pickle.load(f)
            else:
                cache_path.unlink()
        
        print(f"💾 Cache MISS for {func_name}")
        return None
    
    def set(self, func_name: str, data: Any, *args, **kwargs) -> None:
        """Store data in cache"""
        cache_key = self._get_cache_key(func_name, *args, **kwargs)
        cache_path = self._get_cache_path(cache_key)
        
        with open(cache_path, 'wb') as f:
            pickle.dump(data, f)
        print(f"💾 Cached data for {func_name}")
    
    def clear(self, func_name: Optional[str] = None) -> int:
        """Clear cache"""
        count = 0
        for cache_file in self.cache_dir.glob("*.pkl"):
            if func_name is None or cache_file.stem.startswith(func_name):
                cache_file.unlink()
                count += 1
        print(f"🗑️ Cleared {count} cache files")
        return count
    
    def cache_decorator(self, func):
        """Decorator for automatic caching"""
        from functools import wraps
        
        @wraps(func)
        def wrapper(*args, **kwargs):
            # Skip 'self' parameter for cache key
            func_name = func.__name__
            
            # Create cache key from function name and arguments (excluding self)
            if args and hasattr(args[0], '__class__'):
                # It's a method call, skip the 'self' argument
                cache_args = args[1:]
            else:
                cache_args = args
            
            cache_key = self._get_cache_key(func_name, *cache_args, **kwargs)
            cache_path = self._get_cache_path(cache_key)
            
            # Check if cached
            if cache_path.exists():
                age = datetime.now() - datetime.fromtimestamp(cache_path.stat().st_mtime)
                if age < self.cache_duration:
                    with open(cache_path, 'rb') as f:
                        print(f"💾 Cache HIT for {func_name}")
                        return pickle.load(f)
                else:
                    cache_path.unlink()
            
            # Execute function
            print(f"💾 Cache MISS for {func_name}")
            result = func(*args, **kwargs)
            
            # Cache result
            with open(cache_path, 'wb') as f:
                pickle.dump(result, f)
            print(f"💾 Cached data for {func_name}")
            
            return result
        
        return wrapper

cache_manager = CacheManager()
print("✅ Cache Manager ready (with decorator)")

✅ Cache Manager ready (with decorator)


### Complete Data Pipeline: Putting It All Together

# Simplified Data Pipeline (more reliable)

In [ ]:
class SimpleDataPipeline:
    """Simplified pipeline that focuses on Yahoo Finance only"""
    
    def __init__(self):
        self.cleaner = DataCleaner()
        self.validator = DataValidator()
        
    def fetch_sp500_data(self, start_date: str = None, end_date: str = None) -> pd.DataFrame:
        """Simple data fetch from Yahoo Finance"""
        start_date = start_date or '2010-01-01'
        end_date = end_date or datetime.now().strftime('%Y-%m-%d')
        
        print(f"📊 Fetching S&P 500 data from {start_date} to {end_date}...")
        
        try:
            # Try with SPY first (more reliable)
            ticker = yf.Ticker("SPY")
            data = ticker.history(start=start_date, end=end_date)
            
            if data.empty:
                # Try with ^GSPC as fallback
                print("   SPY returned empty, trying ^GSPC...")
                ticker = yf.Ticker("^GSPC")
                data = ticker.history(start=start_date, end=end_date)
            
            if not data.empty:
                # Clean column names
                data.columns = [col.lower() for col in data.columns]
                
                # Add returns
                data['returns'] = data['close'].pct_change()
                data['log_returns'] = np.log(data['close'] / data['close'].shift(1))
                
                print(f"✅ Fetched {len(data)} days of data")
                return data
            else:
                print("❌ No data returned from Yahoo Finance")
                return pd.DataFrame()
                
        except Exception as e:
            print(f"❌ Error: {e}")
            return pd.DataFrame()
    
    def run_complete_pipeline(self, save_processed: bool = True):
        """Run simplified pipeline"""
        print("\n" + "="*60)
        print("STARTING SIMPLIFIED DATA PIPELINE")
        print("="*60 + "\n")
        
        # Fetch data
        raw_data = self.fetch_sp500_data()
        
        if raw_data.empty:
            raise ValueError("No data collected!")
        
        # Clean data
        print("\n" + "-"*40)
        cleaned_data = self.cleaner.clean_sp500_data(raw_data)
        
        # Validate
        print("\n" + "-"*40)
        quality_report = self.validator.generate_quality_report(cleaned_data)
        
        # Save
        if save_processed and not cleaned_data.empty:
            output_dir = Path('../../data/processed')
            output_dir.mkdir(parents=True, exist_ok=True)
            output_path = output_dir / 'sp500_processed.csv'
            cleaned_data.to_csv(output_path)
            print(f"\n💾 Saved to {output_path}")
        
        print("\n" + "="*60)
        print("PIPELINE COMPLETE")
        print("="*60 + "\n")
        
        return cleaned_data, quality_report

# Create simplified pipeline
simple_pipeline = SimpleDataPipeline()
print("✅ Simplified pipeline ready")

# Run it
data, report = simple_pipeline.run_complete_pipeline(save_processed=True)

# Display results
if not data.empty:
    print(f"\n📊 Success! Got {len(data)} rows of data")
    print(f"📅 Date range: {data.index[0]} to {data.index[-1]}")
    print(f"\n📈 Sample:")
    print(data[['close', 'volume', 'returns']].head())
else:
    print("\n❌ Still no data. Please check your internet connection.")

✅ Simplified pipeline ready

STARTING SIMPLIFIED DATA PIPELINE

📊 Fetching S&P 500 data from 2010-01-01 to 2026-04-10...
✅ Fetched 4091 days of data

----------------------------------------
🧹 Starting data cleaning pipeline...
  - Removed 0 duplicate rows
  - Capped 82 outliers in open
  - Capped 82 outliers in high
  - Capped 82 outliers in low
  - Capped 82 outliers in close
  - Capped 82 outliers in volume
  - Capped 41 outliers in dividends
  - Fixed 1 close range violations
✅ Cleaning complete. Shape: (4091, 10)

----------------------------------------
📋 Generating quality report...
  - Quality score: 100.0/100
  - Valid for modeling: True
  - Total rows: 4091
  - Total columns: 10

💾 Saved to ..\..\data\processed\sp500_processed.csv

PIPELINE COMPLETE


📊 Success! Got 4091 rows of data
📅 Date range: 2010-01-04 00:00:00-05:00 to 2026-04-09 00:00:00-04:00

📈 Sample:
                               close       volume   returns
Date                                                   

# Create Data Pipeline (Run this before Cell 10)

In [ ]:
from sklearn.impute import SimpleImputer

class DataPipeline:
    """End-to-end data pipeline"""
    
    def __init__(self):
        self.cleaner = DataCleaner()
        self.validator = DataValidator()
        self.yahoo_collector = YahooFinanceCollector()
        
    def fetch_all_data(self, start_date: str = None, end_date: str = None):
        """Fetch data from Yahoo Finance"""
        start_date = start_date or DataConfig.START_DATE
        end_date = end_date or DataConfig.END_DATE
        
        print("🚀 Starting data collection...")
        
        # Yahoo Finance data
        sp500_data = self.yahoo_collector.fetch_daily_data(start_date, end_date)
        
        if sp500_data.empty:
            raise ValueError("No S&P 500 data collected!")
        
        print("✅ Data collection complete")
        return {'sp500_raw': sp500_data}
    
    def run_complete_pipeline(self, save_processed: bool = True):
        """Run complete pipeline"""
        print("\n" + "="*60)
        print("STARTING COMPLETE DATA PIPELINE")
        print("="*60 + "\n")
        
        # Collect data
        raw_data = self.fetch_all_data()
        
        if 'sp500_raw' not in raw_data or raw_data['sp500_raw'].empty:
            raise ValueError("No S&P 500 data collected!")
        
        # Clean data
        print("\n" + "-"*40)
        cleaned_data = self.cleaner.clean_sp500_data(raw_data['sp500_raw'])
        
        # Validate data
        print("\n" + "-"*40)
        quality_report = self.validator.generate_quality_report(cleaned_data)
        
        # Save processed data
        if save_processed:
            output_dir = Path('../../data/processed')
            output_dir.mkdir(parents=True, exist_ok=True)
            
            output_path = output_dir / 'sp500_processed.csv'
            cleaned_data.to_csv(output_path)
            print(f"\n💾 Saved processed data to {output_path}")
        
        print("\n" + "="*60)
        print("DATA PIPELINE COMPLETE")
        print("="*60 + "\n")
        
        return cleaned_data, quality_report

# Create pipeline instance
pipeline = DataPipeline()
print("✅ Data Pipeline ready")

✅ Data Pipeline ready


# Re-run pipeline

In [ ]:
data, report = pipeline.run_complete_pipeline(save_processed=True)

# Verify data
if not data.empty:
    print(f"\n✅ SUCCESS! Got {len(data)} rows of data")
    print(f"📅 Date range: {data.index[0]} to {data.index[-1]}")
    print(f"\n📈 Sample data:")
    print(data[['close', 'volume', 'returns']].head(10))
    print(f"\n📊 Data shape: {data.shape}")
    print(f"\n✅ Data is now OK!")
else:
    print(f"\n❌ Still empty. Shape: {data.shape}")


STARTING COMPLETE DATA PIPELINE

🚀 Starting data collection...
📊 Fetching S&P 500 data from Yahoo Finance...
✅ Fetched 4091 days of data
✅ Data collection complete

----------------------------------------
🧹 Starting data cleaning pipeline...
  - Removed 0 duplicate rows
  - Capped 82 outliers in open
  - Capped 82 outliers in high
  - Capped 82 outliers in low
  - Capped 82 outliers in close
  - Capped 82 outliers in volume
  - Capped 41 outliers in dividends
  - Fixed 1 close range violations
✅ Cleaning complete. Shape: (4091, 11)

----------------------------------------
📋 Generating quality report...
  - Quality score: 100.0/100
  - Valid for modeling: True
  - Total rows: 4091
  - Total columns: 11

💾 Saved processed data to ..\..\data\processed\sp500_processed.csv

DATA PIPELINE COMPLETE


✅ SUCCESS! Got 4091 rows of data
📅 Date range: 2010-01-04 00:00:00-05:00 to 2026-04-09 00:00:00-04:00

📈 Sample data:
                               close       volume   returns
Date          

# Standalone Yahoo Finance Test

In [ ]:
print("🧪 STANDALONE YAHOO FINANCE TEST")
print("="*50)

import yfinance as yf

# Test 1: Basic connection
print("\nTest 1: Checking Yahoo Finance connection...")
try:
    sp500 = yf.Ticker("^GSPC")
    print("   ✅ Ticker object created")
except Exception as e:
    print(f"   ❌ Failed: {e}")

# Test 2: Get recent data
print("\nTest 2: Fetching recent data...")
try:
    recent = sp500.history(period="5d")
    if not recent.empty:
        print(f"   ✅ Success! Got {len(recent)} days of data")
        print(f"   Latest date: {recent.index[-1]}")
        print(f"   Latest close: ${recent['Close'].iloc[-1]:.2f}")
    else:
        print("   ❌ No data returned")
except Exception as e:
    print(f"   ❌ Failed: {e}")

# Test 3: Get historical data
print("\nTest 3: Fetching historical data (2024)...")
try:
    historical = sp500.history(start="2024-01-01", end="2024-12-31")
    if not historical.empty:
        print(f"   ✅ Success! Got {len(historical)} trading days in 2024")
    else:
        print("   ❌ No historical data returned")
except Exception as e:
    print(f"   ❌ Failed: {e}")

print("\n" + "="*50)

if recent.empty and historical.empty:
    print("\n⚠️ Yahoo Finance is not returning data. Possible causes:")
    print("  1. Rate limiting - wait a few minutes and try again")
    print("  2. VPN/Proxy blocking - try disabling")
    print("  3. yfinance version - run: pip install --upgrade yfinance")
    print("  4. Alternative: Use 'SPY' instead of '^GSPC'")
    
    # Try alternative ticker
    print("\n🔄 Trying with SPY ETF instead...")
    spy = yf.Ticker("SPY")
    spy_data = spy.history(period="5d")
    if not spy_data.empty:
        print(f"   ✅ SPY works! Got {len(spy_data)} days")
        print("   Use 'SPY' instead of '^GSPC' in your config")
else:
    print("\n✅ Yahoo Finance is working correctly!")

🧪 STANDALONE YAHOO FINANCE TEST

Test 1: Checking Yahoo Finance connection...
   ✅ Ticker object created

Test 2: Fetching recent data...
   ✅ Success! Got 5 days of data
   Latest date: 2026-04-09 00:00:00-04:00
   Latest close: $6824.66

Test 3: Fetching historical data (2024)...
   ✅ Success! Got 251 trading days in 2024


✅ Yahoo Finance is working correctly!


### Run the Pipeline

In [103]:
# Execute the pipeline
if __name__ == "__main__":
    
    # Initialize pipeline
    pipeline = DataPipeline()
    
    # Run complete pipeline
    data, report = pipeline.run_complete_pipeline(save_processed=True)
    
    # Display results
    print("\n📊 DATA SUMMARY:")
    print(f"  - Shape: {data.shape}")
    
    if not data.empty:
        print(f"  - Date range: {data.index[0]} to {data.index[-1]}")
    else:
        print("  - No data available!")

    print(f"  - Columns: {list(data.columns)}")
    
    print("\n📈 SAMPLE DATA:")
    print(data.head(10))
    
    print("\n📊 DATA STATISTICS:")
    print(data.describe())
    
    print("\n✅ Pipeline execution successful!")



STARTING COMPLETE DATA PIPELINE

🚀 Starting data collection...
📊 Fetching S&P 500 data from Yahoo Finance...
✅ Fetched 4091 days of data
✅ Data collection complete

----------------------------------------
🧹 Starting data cleaning pipeline...
  - Removed 0 duplicate rows
  - Capped 82 outliers in open
  - Capped 82 outliers in high
  - Capped 82 outliers in low
  - Capped 82 outliers in close
  - Capped 82 outliers in volume
  - Capped 41 outliers in dividends
  - Fixed 1 close range violations
✅ Cleaning complete. Shape: (4091, 11)

----------------------------------------
📋 Generating quality report...
  - Quality score: 100.0/100
  - Valid for modeling: True
  - Total rows: 4091
  - Total columns: 11

💾 Saved processed data to ..\..\data\processed\sp500_processed.csv

DATA PIPELINE COMPLETE


📊 DATA SUMMARY:
  - Shape: (4091, 11)
  - Date range: 2010-01-04 00:00:00-05:00 to 2026-04-09 00:00:00-04:00
  - Columns: ['open', 'high', 'low', 'close', 'volume', 'dividends', 'stock splits'

### Quick Test with Real Data

In [104]:
# Quick test (run this cell first to verify everything works)
def quick_test():
    """Quick test function to verify all components work"""
    
    print("🧪 Running quick test...\n")
    
    # Test Yahoo Finance
    yahoo = YahooFinanceCollector()
    test_data = yahoo.fetch_daily_data('2024-01-01', '2024-01-10')
    
    if not test_data.empty:
        print(f"✅ Yahoo Finance: {len(test_data)} rows")
    
    # Test Cleaner
    cleaner = DataCleaner()
    cleaned = cleaner.clean_sp500_data(test_data)
    print(f"✅ Data Cleaner: {len(cleaned)} rows after cleaning")
    
    # Test Validator
    validator = DataValidator()
    report = validator.generate_quality_report(cleaned)
    print(f"✅ Data Validator: Quality score {report['quality_score']:.1f}")
    
    # Test Cache
    cache = CacheManager()
    cache.set("test", "test_data")
    cached = cache.get("test")
    print(f"✅ Cache Manager: {'Working' if cached else 'Failed'}")
    cache.clear()
    
    print("\n🎉 All components working correctly!")

# Run quick test
quick_test()

🧪 Running quick test...

📊 Fetching S&P 500 data from Yahoo Finance...
✅ Fetched 6 days of data
✅ Yahoo Finance: 6 rows
🧹 Starting data cleaning pipeline...
  - Removed 0 duplicate rows
  - Dropped columns: ['volatility']
  - Capped 2 outliers in open
  - Capped 2 outliers in high
  - Capped 2 outliers in low
  - Capped 2 outliers in close
  - Capped 2 outliers in volume
✅ Cleaning complete. Shape: (6, 10)
✅ Data Cleaner: 6 rows after cleaning
📋 Generating quality report...
  - Quality score: 100.0/100
  - Valid for modeling: True
  - Total rows: 6
  - Total columns: 10
✅ Data Validator: Quality score 100.0
💾 Cached data for test
💾 Cache HIT for test
✅ Cache Manager: Working
🗑️ Cleared 1 cache files

🎉 All components working correctly!


# Quick Test

In [ ]:

def quick_test():
    """Quick test function"""
    print("🧪 Running quick test...\n")
    
    # Test Yahoo Finance
    test_data = yahoo_collector.fetch_daily_data('2024-01-01', '2024-01-10')
    if not test_data.empty:
        print(f"✅ Yahoo Finance: {len(test_data)} rows")
    
    # Test Cleaner
    cleaned = cleaner.clean_sp500_data(test_data)
    print(f"✅ Data Cleaner: {len(cleaned)} rows")
    
    # Test Validator
    report = validator.generate_quality_report(cleaned)
    print(f"✅ Data Validator: Score {report['quality_score']:.1f}")
    
    # Test Cache
    cache_manager.set("test", "test_data")
    cached = cache_manager.get("test")
    print(f"✅ Cache Manager: {'Working' if cached else 'Failed'}")
    cache_manager.clear()
    
    print("\n🎉 All components working!")

# Run the test
quick_test()

🧪 Running quick test...

📊 Fetching S&P 500 data from Yahoo Finance...
✅ Fetched 6 days of data
✅ Yahoo Finance: 6 rows
🧹 Starting data cleaning pipeline...
  - Removed 0 duplicate rows
  - Dropped columns: ['volatility']
  - Capped 2 outliers in open
  - Capped 2 outliers in high
  - Capped 2 outliers in low
  - Capped 2 outliers in close
  - Capped 2 outliers in volume
✅ Cleaning complete. Shape: (6, 10)
✅ Data Cleaner: 6 rows
📋 Generating quality report...
  - Quality score: 100.0/100
  - Valid for modeling: True
  - Total rows: 6
  - Total columns: 10
✅ Data Validator: Score 100.0
💾 Cached data for test
💾 Cache HIT for test
✅ Cache Manager: Working
🗑️ Cleared 1 cache files

🎉 All components working!
